In [1]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## 1、下载数据

In [2]:
from datasets import load_dataset
ds = load_dataset("llamafactory/alpaca_zh", cache_dir="../data",split = "train")

In [3]:
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 51155
})

In [4]:
ds = ds.train_test_split(test_size=0.8,seed = 42)
ds

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 40924
    })
})

In [5]:
ds['train'][5]

{'instruction': '解释气候变化对环境的两个影响',
 'input': '',
 'output': '气候变化对环境有几个影响。其中一个影响是全球平均气温上升，导致冰川融化、极端天气事件和海平面上升。另一个影响是由极端天气条件和升温引起的自然栖息地破坏导致的生物多样性减少。'}

## 2、数据预处理

In [6]:
# 模型下载
# from modelscope import snapshot_download
# model_dir = snapshot_download('Langboat/bloom-1b4-zh', cache_dir="./models")

In [7]:
model_path = r"D:\project\transformer\NLP_transformer\transformer\models\qwen\Qwen2-0___5B"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Qwen2TokenizerFast(name_or_path='D:\project\transformer\NLP_transformer\transformer\models\qwen\Qwen2-0___5B', vocab_size=151643, model_max_length=32768, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [9]:
def process_func(datasets, tokenizer, max_length=256):
    """
    处理输入示例，合并用户的指令和输入字段，然后对输入和输出进行分词。
    该函数准备好训练自回归语言模型的数据格式。

    参数：
    - datasets (dict): 包含 'instruction'、'input' 和 'output' 字段的字典。
    - tokenizer (AutoTokenizer): 用于分词的 tokenizer。
    - max_length (int): 序列的最大长度。

    返回：
    - dict: 准备好的训练数据，包括 tokenized 的输入和标签。
    """
    # 1. 生成输入部分，包括 "Human: " 和 "Assistant: " 标签
    if datasets['input'].strip() == "":
        combined_input = "用户: " + datasets['instruction'] + "\n\n 机器人: "
    else:
        combined_input = "用户: " + datasets['instruction'] + "\n" + datasets['input'] + "\n\n 机器人: "

    # 2. 生成输出部分，并在最后加上终止标记
    output_text = datasets["output"] + tokenizer.eos_token  # 加上 eos_token 标记回复的结束

    # 3. 合并输入和输出作为模型的完整输入序列
    full_input = combined_input + output_text

    # 4. 对完整输入进行分词处理，确保生成 input_ids 和 attention_mask
    encodings = tokenizer(
        full_input, 
        max_length=max_length, 
        truncation=True, 
        padding="max_length", 
        return_tensors='pt'
    )

    # 5. 复制 input_ids 来生成标签
    labels = encodings["input_ids"].clone()

    # 6. 计算用户输入部分的长度（即需要忽略的部分）
    user_input_len = len(tokenizer(combined_input, truncation=True, max_length=max_length)["input_ids"])

    # 7. 将用户输入部分的标签设置为 -100，忽略这部分的损失
    labels[:, :user_input_len] = -100

    # 8. 返回 input_ids、attention_mask 和 labels，用于训练模型
    return {
        'input_ids': encodings['input_ids'].squeeze(0),  # 输入序列
        'attention_mask': encodings['attention_mask'].squeeze(0),  # 注意力掩码
        'labels': labels.squeeze(0)  # 标签，忽略用户输入部分
    }


In [10]:
# 假设 ds 是你的数据集，并且 tokenizer 已经定义
tokenized_ds = ds.map(
    process_func,
    remove_columns=['instruction', 'input', 'output'],  # 删除不需要的原始列
    fn_kwargs={'tokenizer': tokenizer},  # 将 tokenizer 传递给 process_func
)

tokenized_ds

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 10231
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 40924
    })
})

## 3、创建模型

In [11]:
import torch
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    trust_remote_code=True, 
    low_cpu_mem_usage=True, 
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_use_double_quant=True)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


### 3.1 设置微调参数

In [12]:
model.dtype

torch.bfloat16

In [13]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(task_type=TaskType.CAUSAL_LM)
config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=8, target_modules=None, lora_alpha=8, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))

### 3.2 重新载入模型

In [14]:
model = get_peft_model(model, config)
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear4bit(in_features=896,

In [15]:
for name, parameter in model.named_parameters():
    print(name, parameter.shape,parameter.dtype)

base_model.model.model.embed_tokens.weight torch.Size([151936, 896]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight torch.Size([401408, 1]) torch.uint8
base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias torch.Size([896]) torch.bfloat16
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([8, 896]) torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([896, 8]) torch.float32
base_model.model.model.layers.0.self_attn.k_proj.weight torch.Size([57344, 1]) torch.uint8
base_model.model.model.layers.0.self_attn.k_proj.bias torch.Size([128]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight torch.Size([57344, 1]) torch.uint8
base_model.model.model.layers.0.self_attn.v_proj.base_layer.bias torch.Size([128]) torch.bfloat16
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([8, 896]) torch.float32
base_model.model.mo

### 3.3 查看参数量

In [16]:
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 4、创建训练参数

In [17]:
args = TrainingArguments(
    output_dir="./models_for_chatbot_LoRA",
    per_device_train_batch_size=4, # 每个设备训练批次大小为4
    gradient_accumulation_steps=8,  # 每4步累计一次
    per_device_eval_batch_size=32, # 每个设备评估批次大小为32
    save_strategy="epoch",  # 每个epoch保存一次
    logging_steps=20, # 每个20步打印一次
    num_train_epochs=1, # 训练1个epoch
    report_to=['tensorboard'],
    optim="paged_adamw_32bit", # 使用paged_adamw_32bit优化器
)

## 5、创建训练器并训练

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"], 
    eval_dataset=tokenized_ds["test"],     
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True)
)

d:\program\anaconda3\envs\transformers\Lib\site-packages\accelerate\accelerator.py:457: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [19]:
trainer.train()

  0%|          | 0/319 [00:00<?, ?it/s]

{'loss': 14.1777, 'grad_norm': 16.091806411743164, 'learning_rate': 4.686520376175549e-05, 'epoch': 0.06}
{'loss': 12.5486, 'grad_norm': 17.48172950744629, 'learning_rate': 4.373040752351097e-05, 'epoch': 0.13}
{'loss': 10.247, 'grad_norm': 22.955921173095703, 'learning_rate': 4.059561128526646e-05, 'epoch': 0.19}
{'loss': 7.5549, 'grad_norm': 32.386146545410156, 'learning_rate': 3.746081504702195e-05, 'epoch': 0.25}
{'loss': 4.0087, 'grad_norm': 27.105222702026367, 'learning_rate': 3.4326018808777435e-05, 'epoch': 0.31}
{'loss': 1.6055, 'grad_norm': 7.97143030166626, 'learning_rate': 3.119122257053292e-05, 'epoch': 0.38}
{'loss': 0.7106, 'grad_norm': 0.8736119866371155, 'learning_rate': 2.8056426332288405e-05, 'epoch': 0.44}
{'loss': 0.5778, 'grad_norm': 2.2503058910369873, 'learning_rate': 2.4921630094043887e-05, 'epoch': 0.5}
{'loss': 0.5572, 'grad_norm': 0.24314966797828674, 'learning_rate': 2.1786833855799376e-05, 'epoch': 0.56}
{'loss': 0.5212, 'grad_norm': 0.16364894807338715, '

Checkpoint destination directory ./models_for_chatbot_LoRA\checkpoint-319 already exists and is non-empty. Saving will proceed but saved results may be invalid.


{'train_runtime': 701.9434, 'train_samples_per_second': 14.575, 'train_steps_per_second': 0.454, 'train_loss': 3.4944546260056453, 'epoch': 1.0}


TrainOutput(global_step=319, training_loss=3.4944546260056453, metrics={'train_runtime': 701.9434, 'train_samples_per_second': 14.575, 'train_steps_per_second': 0.454, 'train_loss': 3.4944546260056453, 'epoch': 1.0})